# Demo · extracción LLM de un PDF de la Gaceta

Este notebook permite elegir un PDF propio o una norma de la Gaceta y ver, paso a paso, cómo el LLM lo convierte en un registro `GacetaNorma`.

**Qué muestra**

- PDF: edición, norma, páginas, SHA-256 y preview.
- Cache: si el PDF ya fue extraído, no se llama a la API.
- Extracción: prompt, respuesta cruda, JSON validado y campos.
- Costo: tiempo, tokens y precio de cada corrida.
- Comparación: historial de corridas con GLM y DeepSeek.

**Importante**

- No escribe en `data/`: no modifica `llm_manifest.json` ni `gaceta_llm.csv`.
- Solo llama a la API si marcás **Confirmo gasto de API**.
- Las claves se leen de `.env`; nunca se imprimen.

## 1. Setup

Carga el entorno, el cache real y los clientes de los dos proveedores.

Requiere `ipywidgets` (está en el grupo `dev`; correr `uv sync`).

El `.env` del proyecto tiene prioridad sobre variables viejas del kernel. Si ves un 401, presioná **Probar credenciales** y, si persiste, reiniciá el kernel.

In [1]:
import hashlib
import html
import json
import os
import sys
import time
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import pymupdf
from dotenv import dotenv_values
from IPython.display import HTML, Image, JSON, display
from pydantic import ValidationError

# Rutas: funciona con cwd=modelado/ o cwd=raíz.
MODELADO = (Path.cwd() if (Path.cwd() / "metricas_reporte.json").exists()
            else Path.cwd() / "modelado")
FP = MODELADO.parent
DATA = FP / "data"
OUT = MODELADO / "outputs" / "llm_demo"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(FP))
# El .env del proyecto tiene prioridad en esta demo: evita claves viejas
# que hayan quedado en el entorno del kernel.
ENV = dotenv_values(FP / ".env")
for _name in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "LLM_MAX_COST_USD",
              "DEEPSEEK_API_KEY", "DEEPSEEK_BASE_URL", "DEEPSEEK_MODEL"):
    if ENV.get(_name):
        os.environ[_name] = ENV[_name]
    elif _name in ("LLM_BASE_URL", "LLM_MODEL",
                   "DEEPSEEK_BASE_URL", "DEEPSEEK_MODEL"):
        os.environ.pop(_name, None)

from processors.llm_client import (CostCapExceeded, LLMUnavailable,
                                   build_client)
from processors.llm_file_processor import SYSTEM_PROMPT, _user_prompt
from processors.llm_manifest import CACHE_VERSION, LLMManifest, sha256_file
from processors.results import GacetaNorma

pd.set_option("display.max_colwidth", 100)
MANIFEST = LLMManifest(DATA / "processed" / "llm_manifest.json")
PDFS = sorted((DATA / "gaceta").glob("*/norma_*.pdf"))
if not PDFS:
    raise SystemExit("no hay PDFs en data/gaceta")
PDF_DEFAULT = PDFS[0]
print(f"PDFs disponibles: {len(PDFS)} · cache: {len(MANIFEST.entries)} entradas")

PDFs disponibles: 170 · cache: 169 entradas


### Campos que extrae el LLM

El esquema se valida con Pydantic. Si el modelo devuelve un tag fuera de la taxonomía o una relevancia inválida, el intento se rechaza y se reintenta.

In [2]:
props = GacetaNorma.model_json_schema()["properties"]
display(pd.DataFrame([
    {"Campo": k, "Descripción": v.get("description", "")}
    for k, v in props.items()
]))

,Campo,Descripción
0,norma_id,ID de la norma (se sobreescribe con el del path)
1,edicion,Edición de la Gaceta (se sobreescribe con la del path)
2,instrumento,"Tipo de instrumento: Decreto Supremo, Resolución Ministerial, Ley, ..."
3,entidad_emisora,Entidad que emite la norma
4,fecha_publicacion,Fecha de publicación (YYYY-MM-DD)
5,fecha_vigencia,Fecha de vigencia (YYYY-MM-DD)
6,monto_usd,Monto explícito en dólares americanos (número plano); null si no hay o es otra moneda
7,moneda,USD | BOB | null — moneda del monto mencionado
8,resumen_1l,Resumen en una línea de qué trata la norma
9,tags,Tags FX aplicables de TAGS_FX


## 2. Elegir el PDF

Podés subir un PDF propio, escribir un `norma_id` o una ruta, o dejar vacío para usar un PDF de ejemplo.

- Prioridad: **PDF subido** → **texto** → **PDF de ejemplo**.
- El PDF subido se guarda en `modelado/outputs/llm_demo/`.
- Para un PDF custom, el cache probablemente sea miss: marcá **Confirmo gasto de API**.

In [3]:
LABELS = {"deepseek": "DeepSeek", "zai": "GLM (Z.ai)"}
CLIENTES = {k: build_client(k) for k in ("deepseek", "zai")}

filas = []
for key, client in CLIENTES.items():
    filas.append({
        "Proveedor": LABELS[key],
        "Modelo": client.model,
        "Base URL": client.base_url,
        "API key": "ok" if client.available else f"falta {client.api_key_env}",
        "Tope USD": f"{client.max_cost:.2f}",
    })
display(pd.DataFrame(filas))

upload = widgets.FileUpload(accept=".pdf", multiple=False, description="PDF propio")
texto = widgets.Text(
    description="norma/ruta",
    placeholder="281267 o data/gaceta/156ESP/norma_281267.pdf",
    layout=widgets.Layout(width="100%"),
)
proveedor = widgets.Dropdown(
    options=[("DeepSeek", "deepseek"), ("GLM (Z.ai)", "zai")],
    value="deepseek",
    description="Proveedor",
)
forzar = widgets.Checkbox(value=False, description="Forzar llamada")
confirmar = widgets.Checkbox(value=False, description="Confirmo gasto API")
btn_ver = widgets.Button(description="Inspeccionar PDF")
btn_run = widgets.Button(description="Extraer", button_style="primary")
btn_keys = widgets.Button(description="Probar credenciales")
btn_limpiar = widgets.Button(description="Limpiar")
progress = widgets.IntProgress(value=0, min=0, max=100, description="0%",
                               bar_style="info")
estado = widgets.HTML("<i>Elegí un PDF y presioná Inspeccionar.</i>")
salida = widgets.Output()
historial = []


def _fmt_int(value):
    return "—" if value is None else f"{int(value):,}".replace(",", " ")


def _fmt_money(value):
    return "—" if value is None else f"$ {value:.6f}".replace(".", ",")


def _fmt_s(value):
    return "—" if value is None else f"{value:.2f} s".replace(".", ",")


def _set_progress(pct, msg, style="info"):
    progress.value = int(pct)
    progress.description = f"{pct}%"
    progress.bar_style = style
    estado.value = (
        "<div style='padding:6px 0;color:#334155;font-size:0.95em'>"
        f"{msg}</div>"
    )
    time.sleep(0.05)


def _detail(title, body):
    display(HTML(
        f"<details style='margin:8px 0'><summary style='cursor:pointer;"
        f"font-weight:600'>{html.escape(title)}</summary>"
        f"<pre style='white-space:pre-wrap;font-size:12px;background:#f8fafc;"
        f"padding:10px;border:1px solid #e2e8f0;border-radius:8px'>"
        f"{html.escape(body or '')}</pre></details>"
    ))


def _show_record(record):
    display(pd.DataFrame({
        "Campo": list(record.keys()),
        "Valor": ["—" if v in (None, "") else str(v) for v in record.values()],
    }))


def _uploaded_pdf():
    val = upload.value
    if not val:
        return None
    if isinstance(val, dict):
        name, info = next(iter(val.items()))
    else:
        item = val[0]
        name, info = item["name"], item
    content = bytes(info["content"])
    if len(content) > 30 * 1024 * 1024:
        raise ValueError("el PDF supera 30 MB")
    if b"%PDF" not in content[:1024]:
        raise ValueError("el archivo no parece un PDF (%PDF)")
    safe = "".join(ch if ch.isalnum() or ch in "._-" else "_"
                   for ch in Path(name).stem)[:80] or "subido"
    digest = hashlib.sha256(content).hexdigest()[:8]
    path = OUT / f"{safe}_{digest}.pdf"
    path.write_bytes(content)
    return path


def _resolve_pdf():
    subido = _uploaded_pdf()
    if subido is not None:
        return subido
    raw = texto.value.strip()
    if raw:
        p = Path(raw).expanduser()
        if not p.is_absolute():
            p = FP / p
        if p.exists() and p.is_file():
            return p
        if raw.isdigit():
            hits = sorted((DATA / "gaceta").glob(f"*/norma_{raw}.pdf"))
            if hits:
                return hits[0]
        hits = sorted((DATA / "gaceta").glob(f"*/{raw}"))
        if hits:
            return hits[0]
        raise FileNotFoundError(f"no encontré {raw!r}")
    return PDF_DEFAULT


def _pdf_meta(pdf):
    with pymupdf.open(pdf) as doc:
        pages = doc.page_count
        text = doc[0].get_text() if pages else ""
        preview = doc[0].get_pixmap(dpi=80).tobytes("png") if pages else None
    return pages, text, preview


def _ids(pdf):
    norma_id = pdf.stem.removeprefix("norma_")
    edicion = pdf.parent.name if pdf.parent.parent.name == "gaceta" else "custom"
    return norma_id, edicion


def _merge_usage(total, usage):
    if not usage:
        return
    for key, value in usage.items():
        if isinstance(value, dict):
            _merge_usage(total.setdefault(key, {}), value)
        elif isinstance(value, (int, float)):
            total[key] = total.get(key, 0) + value


def _tokens(usage):
    usage = usage or {}
    details = usage.get("prompt_tokens_details") or {}
    return {
        "in": usage.get("prompt_tokens"),
        "out": usage.get("completion_tokens"),
        "total": usage.get("total_tokens"),
        "cached": usage.get("prompt_cache_hit_tokens",
                            details.get("cached_tokens", 0)),
        "miss": usage.get("prompt_cache_miss_tokens"),
    }


def _n_images(messages):
    n = 0
    for message in messages:
        content = message.get("content")
        if isinstance(content, list):
            n += sum(1 for item in content
                     if isinstance(item, dict) and item.get("type") == "image_url")
    return n


def _resumen(pdf, key, client):
    sha = sha256_file(pdf)
    hit = MANIFEST.get(sha, client.provider, client.model)
    return sha, hit


def inspeccionar():
    salida.clear_output()
    try:
        pdf = _resolve_pdf()
    except Exception as exc:
        _set_progress(0, f"Error: {exc}", "danger")
        with salida:
            display(HTML(f"<b>Error:</b> {html.escape(str(exc))}"))
        return

    key = proveedor.value
    client = CLIENTES[key]
    sha, hit = _resumen(pdf, key, client)
    pages, text, preview = _pdf_meta(pdf)
    norma_id, edicion = _ids(pdf)

    with salida:
        display(HTML(f"<h4 style='margin:4px 0'>{html.escape(pdf.name)}</h4>"))
        display(pd.DataFrame([{
            "Edición": edicion,
            "Norma": norma_id,
            "Páginas": pages,
            "Tamaño": f"{pdf.stat().st_size / 1024:.0f} KB",
            "SHA-256": sha[:16] + "…",
            "Proveedor": LABELS[key],
            "Modelo": client.model,
            "API key": "ok" if client.available else f"falta {client.api_key_env}",
        }]))
        if preview:
            display(Image(data=preview, width=520))
        if text.strip():
            _detail("Texto de la primera página", text[:1200])
        else:
            display(HTML("<i>La primera página no tiene capa de texto; el LLM la recibe como imagen.</i>"))
        if key == "deepseek" and pages > client.MAX_PAGES:
            display(HTML(
                f"<b>Atención:</b> DeepSeek envía solo {client.MAX_PAGES} "
                f"de {pages} páginas."
            ))
        display(HTML(
            "<b>Cache key:</b> "
            f"<code>{sha}:{client.provider}:{client.model}:{CACHE_VERSION}</code>"
        ))
        if hit:
            _set_progress(0, "Cache hit: no hace falta llamar al LLM.", "info")
            tok_hit = _tokens(hit.get("usage", {}))
            display(pd.DataFrame([{
                "Estado": "cache",
                "Tiempo esta corrida": "0,00 s",
                "Tokens in (original)": _fmt_int(tok_hit["in"]),
                "Tokens out (original)": _fmt_int(tok_hit["out"]),
                "Tokens total (original)": _fmt_int(tok_hit["total"]),
                "Costo original": _fmt_money(hit.get("cost_usd")),
                "Costo esta corrida": "$ 0,000000",
            }]))
            _show_record(hit.get("record", {}))
            _detail("JSON crudo cacheado", hit.get("raw_content", ""))
        else:
            _set_progress(0, "Cache miss: marcá “Confirmo gasto API” y presioná Extraer.", "warning")
            display(HTML("<b>Estado:</b> cache miss"))


def extraer():
    salida.clear_output()
    try:
        pdf = _resolve_pdf()
    except Exception as exc:
        _set_progress(0, f"Error: {exc}", "danger")
        with salida:
            display(HTML(f"<b>Error:</b> {html.escape(str(exc))}"))
        return

    key = proveedor.value
    client = CLIENTES[key]
    sha, hit = _resumen(pdf, key, client)
    norma_id, edicion = _ids(pdf)

    if hit and not forzar.value:
        _set_progress(100, "Cache hit: no se llamó al LLM.", "success")
        tok_hit = _tokens(hit.get("usage", {}))
        with salida:
            display(HTML(f"<h4>Cache hit · {html.escape(pdf.name)}</h4>"))
            display(pd.DataFrame([{
                "Proveedor": LABELS[key],
                "Modelo": hit.get("model") or client.model,
                "Estado": "cache",
                "Tiempo esta corrida": "0,00 s",
                "Tokens in (original)": _fmt_int(tok_hit["in"]),
                "Tokens out (original)": _fmt_int(tok_hit["out"]),
                "Tokens total (original)": _fmt_int(tok_hit["total"]),
                "Costo original": _fmt_money(hit.get("cost_usd")),
                "Costo esta corrida": "$ 0,000000",
            }]))
            _show_record(hit.get("record", {}))
            _detail("JSON crudo cacheado", hit.get("raw_content", ""))
        historial.append({
            "Proveedor": LABELS[key], "Modelo": client.model, "Estado": "cache",
            "Tiempo total": "0,00 s", "Prep": "—", "LLM": "—", "Valid.": "—",
            "Tokens in": _fmt_int(tok_hit["in"]),
            "Tokens out": _fmt_int(tok_hit["out"]),
            "Tokens total": _fmt_int(tok_hit["total"]),
            "Cacheados": _fmt_int(tok_hit["cached"]),
            "Costo": "$ 0,000000",
            "Acumulado": _fmt_money(client.spent),
            "Nota": f"cache hit · original {_fmt_money(hit.get('cost_usd'))}",
        })
        with salida:
            display(pd.DataFrame(historial))
        return

    if not client.available:
        _set_progress(0, f"Falta {client.api_key_env} en .env.", "danger")
        with salida:
            display(HTML(f"<b>Falta</b> <code>{client.api_key_env}</code> en <code>.env</code>."))
        return
    if not confirmar.value:
        _set_progress(0, "Marcá “Confirmo gasto API” para llamar al modelo.", "danger")
        with salida:
            display(HTML("<b>No se hizo la llamada.</b>"))
        return

    pages, _text, _preview = _pdf_meta(pdf)
    _set_progress(5, f"PDF listo: {pages} páginas.", "info")
    t0 = time.perf_counter()
    messages = client.build_messages(
        SYSTEM_PROMPT, _user_prompt(norma_id, edicion), pdf
    )
    t_prep = time.perf_counter() - t0
    n_images = _n_images(messages)
    if key == "deepseek":
        mecanismo = (f"{pages} páginas → {min(pages, client.MAX_PAGES)} imágenes "
                     f"PNG (150 dpi) + prompt de texto")
    else:
        mecanismo = f"{pages} páginas → PDF crudo (bloque file) + prompt de texto"

    _set_progress(55, f"Llamando a {LABELS[key]} / {client.model}…", "info")
    data = None
    info = None
    norma = None
    last_error = None
    status = None
    cost = 0.0
    t_llm = 0.0
    t_val = 0.0
    usage = {}

    for attempt in range(1, 4):
        try:
            t_call = time.perf_counter()
            data, info = client.chat_json(messages)
            t_llm += time.perf_counter() - t_call
            cost += info["cost_usd"]
            _merge_usage(usage, info.get("usage", {}))
            t_check = time.perf_counter()
            norma = GacetaNorma.model_validate(data)
            t_val += time.perf_counter() - t_check
            break
        except CostCapExceeded as exc:
            status, last_error = "llm_cost_cap", exc
            break
        except LLMUnavailable as exc:
            status, last_error = "llm_failed", exc
            break
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            if attempt < 3:
                _set_progress(55, f"Intento {attempt}/3 falló; reintentando…", "warning")
                time.sleep(2 * attempt)

    if norma is None:
        if status is None:
            status = ("schema_invalid"
                      if isinstance(last_error, (ValidationError, ValueError))
                      else "llm_failed")
        _set_progress(100, f"Error: {status}", "danger")
        msg = str(last_error)
        hint = ""
        if "401" in msg or "authentication" in msg.lower() or "unauthorized" in msg.lower():
            hint = (
                "<div style='margin-top:8px;padding:8px;background:#fef2f2;"
                "border:1px solid #fecaca;border-radius:8px'>"
                f"401 · Revisá <code>{client.api_key_env}</code> y "
                f"<code>{client.base_url_env}</code>. Presioná "
                "<b>Probar credenciales</b>; si sigue, reiniciá el kernel."
                "</div>"
            )
        with salida:
            display(HTML(f"<b>Error:</b> {html.escape(msg)[:500]}"))
            display(HTML(f"<b>Estado:</b> {status}"))
            if hint:
                display(HTML(hint))
        historial.append({
            "Proveedor": LABELS[key], "Modelo": client.model, "Estado": status,
            "Tiempo total": "—", "Prep": _fmt_s(t_prep), "LLM": _fmt_s(t_llm),
            "Valid.": _fmt_s(t_val), "Tokens in": "—", "Tokens out": "—",
            "Tokens total": "—", "Cacheados": "—", "Costo": _fmt_money(cost),
            "Acumulado": _fmt_money(client.spent), "Nota": str(last_error)[:80],
        })
        with salida:
            display(pd.DataFrame(historial))
        return

    _set_progress(80, "JSON recibido; validando con Pydantic…", "info")
    t_total = t_prep + t_llm + t_val
    record = {
        **norma.model_dump(),
        "norma_id": norma_id,
        "edicion": edicion,
        "tags": "|".join(norma.tags),
        "model": info.get("model", client.model),
        "cost_usd": round(cost, 6),
        "status": "ok",
        "extracted_date": pd.Timestamp.now(tz="UTC").isoformat(),
    }
    tok = _tokens(usage)
    _set_progress(100, "Listo.", "success")

    with salida:
        display(HTML(f"<h4>Resultado · {LABELS[key]} / {client.model}</h4>"))
        display(pd.DataFrame([{
            "Estado": "ok",
            "Tiempo total": _fmt_s(t_total),
            "Preparación": _fmt_s(t_prep),
            "LLM": _fmt_s(t_llm),
            "Validación": _fmt_s(t_val),
            "Tokens in": _fmt_int(tok["in"]),
            "Tokens out": _fmt_int(tok["out"]),
            "Tokens total": _fmt_int(tok["total"]),
            "Cacheados": _fmt_int(tok["cached"]),
            "Costo": _fmt_money(cost),
            "Acumulado": _fmt_money(client.spent),
            "Tope": _fmt_money(client.max_cost),
        }]))
        display(HTML(f"<b>Mecanismo:</b> {html.escape(mecanismo)}"))
        if key == "deepseek":
            display(HTML(f"<b>Imágenes enviadas:</b> {n_images}"))
        _show_record(record)
        _detail("JSON crudo del modelo", info.get("raw_content", ""))
        display(JSON(data))
        _detail("Prompt del sistema", SYSTEM_PROMPT)
        _detail("Prompt del usuario", _user_prompt(norma_id, edicion))
        _detail("Uso crudo del proveedor",
                json.dumps(usage, ensure_ascii=False, indent=2))

    historial.append({
        "Proveedor": LABELS[key], "Modelo": client.model, "Estado": "ok",
        "Tiempo total": _fmt_s(t_total), "Prep": _fmt_s(t_prep),
        "LLM": _fmt_s(t_llm), "Valid.": _fmt_s(t_val),
        "Tokens in": _fmt_int(tok["in"]), "Tokens out": _fmt_int(tok["out"]),
        "Tokens total": _fmt_int(tok["total"]), "Cacheados": _fmt_int(tok["cached"]),
        "Costo": _fmt_money(cost), "Acumulado": _fmt_money(client.spent),
        "Nota": f"{n_images} imágenes" if key == "deepseek" else "PDF crudo",
    })
    with salida:
        display(pd.DataFrame(historial))


def probar_credenciales(_=None):
    salida.clear_output()
    filas = []
    for key, client in CLIENTES.items():
        fila = {
            "Proveedor": LABELS[key],
            "Modelo": client.model,
            "Base URL": client.base_url,
            "Key": "ok" if client.available else f"falta {client.api_key_env}",
            "Modelos": "—",
            "Estado": "—",
        }
        if client.available:
            try:
                modelos = client._client.models.list()
                fila["Modelos"] = len(modelos.data)
                fila["Estado"] = "ok"
            except Exception as exc:  # noqa: BLE001
                fila["Estado"] = f"error: {str(exc)[:140]}"
        filas.append(fila)
    _set_progress(0, "Credenciales verificadas.", "info")
    with salida:
        display(pd.DataFrame(filas))
        display(HTML(
            "<div style='margin-top:8px;color:#64748b'>"
            "Si una fila falla con 401, revisá la key/Base URL o reiniciá el "
            "kernel para descartar variables viejas.</div>"
        ))


def limpiar(_=None):
    upload.value = ()
    texto.value = ""
    salida.clear_output()
    _set_progress(0, "Elegí un PDF y presioná Inspeccionar.", "info")


btn_ver.on_click(lambda _: inspeccionar())
btn_run.on_click(lambda _: extraer())
btn_keys.on_click(probar_credenciales)
btn_limpiar.on_click(limpiar)

display(HTML(
    "<div style='background:#fffbeb;border:1px solid #fde68a;padding:8px 10px;"
    "border-radius:8px;margin:8px 0'>El PDF subido se guarda en "
    "<code>modelado/outputs/llm_demo/</code>. No se escribe en <code>data/</code>."
    "</div>"
))
display(widgets.VBox([
    upload,
    texto,
    widgets.HBox([proveedor, forzar, confirmar]),
    widgets.HBox([btn_ver, btn_run, btn_keys, btn_limpiar]),
    progress,
    estado,
    salida,
]))
inspeccionar()

,Proveedor,Modelo,Base URL,API key,Tope USD
0,DeepSeek,deepseek-flash,https://api.deepseek.com,ok,2.00
1,GLM (Z.ai),glm-5.3-flash,https://api.z.ai/api/paas/v4/,ok,2.00


## 3. Cómo leer el resultado

- **Estado `cache`**: el PDF ya estaba en `llm_manifest.json`; no se gastó API.
- **Estado `ok`**: se llamó al modelo y el JSON pasó la validación Pydantic.
- **`schema_invalid`**: el modelo respondió algo que no cumple `GacetaNorma`.
- **`llm_failed`**: fallo de red, proveedor o cuenta.

La tabla muestra tiempo de preparación, llamada, validación y total; tokens de entrada, salida, totales y cacheados; y el costo de esta corrida junto con el acumulado y el tope.

El historial queda solo en memoria: si reiniciás el kernel, se borra.

## 4. Qué hace el pipeline real con este resultado

Este notebook **no persiste** nada. El pipeline de producción:

1. calcula el SHA-256 del PDF,
2. busca en `llm_manifest.json`,
3. si no hay hit, llama al LLM y valida,
4. guarda la entrada en el manifest y el registro en `gaceta_llm.csv`.

Para persistir un PDF nuevo, usá `main.py process` o `make process`.